In [ ]:
from datetime import datetime
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Get environment variables
mdb_path = os.getenv("MDB")
lead_type_path = os.getenv("LeadType")

if not mdb_path or not os.path.exists(mdb_path):
    print(f"❌ MDB file path not found or not set in environment: {mdb_path}")
elif not lead_type_path or not os.path.exists(lead_type_path):
    print(f"❌ LeadType definition file path not found or not set in environment: {lead_type_path}")
else:
    print(f"📂 Loading MDB file: {mdb_path}")
    df_mdb = pd.read_excel(mdb_path)

    print(f"📂 Loading LeadType definition file: {lead_type_path}")
    df_lead_def = pd.read_excel(lead_type_path)

    # Required lookup columns from definition file
    lookup_cols = [
        "Sector",
        "Activity Description",
        "Proposal Status",
    ]

    # Validate columns exist in both dataframes
    missing_mdb = [col for col in lookup_cols if col not in df_mdb.columns]
    missing_def = [
        col for col in lookup_cols + ["FileType"] if col not in df_lead_def.columns
    ]

    if missing_mdb:
        print(f"❌ Missing columns in MDB file: {missing_mdb}")
    elif missing_def:
        print(f"❌ Missing columns in LeadType definition file: {missing_def}")
    else:
        # File types to process (excluding 'To be Removed')
        file_types = [
            "Non-Mineral_PotentialLeads",
            "Non-Mineral_ActiveLeads",
            "Mineral_PotentialLeads",
            "Mineral_ActiveLeads",
        ]

        # Drop duplicates in definition rules to prevent unintended row explosion during merge
        df_rules = df_lead_def[df_lead_def["FileType"].isin(file_types)][
            lookup_cols + ["FileType"]
        ].drop_duplicates()

        # Perform inner merge to find matching records based on the four criteria columns
        merged_df = pd.merge(
            df_mdb,
            df_rules,
            on=lookup_cols,
            how="inner",
        )

        # Columns to keep in the final generated files
        columns_to_keep = [
            "Proposal No.",
            "Year",
            "Location",
            "Proposal Status",
            "Project Name",
            "Project Proponent",
            "Sector",
            "Activity Description",
            "Form_No",
            "Date of Submission",
            "Comment",
        ]

        # Output directory for the new files (defaults to the same directory as MDB file if SILVER not set)
        output_dir = os.getenv("SILVERS")

        # Create a separate file for each FileType status, removing old ones if they exist
        for f_type in file_types:
            output_file_name = f"{f_type}.xlsx"
            output_file_path = os.path.join(output_dir, output_file_name)

            # Drop the old file if it already exists to ensure a fresh creation
            if os.path.exists(output_file_path):
                try:
                    os.remove(output_file_path)
                    print(f"🗑️ Removed existing file: {output_file_name}")
                except Exception as e:
                    print(f"⚠️ Could not remove existing file {output_file_name}: {e}")

            df_subset = merged_df[merged_df["FileType"] == f_type].copy()

            # Filter columns to keep only those present in the dataframe to avoid KeyErrors
            available_cols_to_keep = [col for col in columns_to_keep if col in df_subset.columns]
            df_subset = df_subset[available_cols_to_keep]

            df_subset.to_excel(output_file_path, index=False)
            print(
                f"✅ Created fresh file '{output_file_name}' with {len(df_subset)} records at: {output_file_path}"
            )

        print("\n🎉 All four lead files generated successfully!")

📂 Loading MDB file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\MasterDB.xlsx
📂 Loading LeadType definition file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\4 - Control Lists\LeadType.xlsx
🗑️ Removed existing file: Non-Mineral_PotentialLeads.xlsx
✅ Created fresh file 'Non-Mineral_PotentialLeads.xlsx' with 3931 records at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\Non-Mineral_PotentialLeads.xlsx
🗑️ Removed existing file: Non-Mineral_ActiveLeads.xlsx
✅ Created fresh file 'Non-Mineral_ActiveLeads.xlsx' with 3522 records at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\Non-Mineral_ActiveLeads.xlsx
🗑️ Removed existing file: Mineral_PotentialLeads.xlsx
✅ Created fresh file 'Mineral_PotentialLeads.xlsx' with 12746 records at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\Mineral_PotentialLeads.xlsx
🗑️ Removed existing file: Mineral_ActiveLeads.xlsx
✅ Created fresh file 'Mineral_ActiveLeads.xlsx' with 39429 rec